In [1]:
import numpy as np
import matplotlib.pyplot as plt
import surfinBH
import precession as pr
import lal
lal.swig_redirect_standard_output_error(False)

/home/giacomo/Thesis/Entropy-conjecture/.venv/lib/python3.12/site-packages/lalsimulation/_lalsimulation_swig.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal
/home/giacomo/Thesis/Entropy-conjecture/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [8]:
# FUNCTION TO CALCULATE GEOMETRIC ENTROPY
def geometric_entropy(mass, chi_vec):

    if isinstance(chi_vec, np.ndarray) and chi_vec.shape[-1] == 3:
        chi_mag = np.linalg.norm(chi_vec, axis=-1)
    else:
        chi_mag = chi_vec
    
    chi_mag = np.clip(chi_mag, 0.0, 0.9999) 
    
    return (mass**2) * (1.0 + np.sqrt(1.0 - chi_mag**2))

In [9]:
#SFERICAL COORDINATES TO FIX SPIN MODULE AT 1
def spherical_to_cartesian(chi_mag, theta, phi):
    
    chi_mag = np.clip(chi_mag, 0.0, 1.0)
    
    chi_x = chi_mag * np.sin(theta) * np.cos(phi)
    chi_y = chi_mag * np.sin(theta) * np.sin(phi)
    chi_z = chi_mag * np.cos(theta)

    if isinstance(chi_mag, np.ndarray):
        return np.column_stack((chi_x, chi_y, chi_z))
    return np.array([chi_x, chi_y, chi_z])

In [ ]:
fit_remnant = surfinBH.LoadFits('NRSur7dq4EmriRemnant')

Loaded NRSur7dq4EmriRemnant fit.
Loaded NRSur7dq4 model


In [15]:
# INITIAL PARAMETERS AND ENTROPY
q = 2.0 
m1 = q / (1.0 + q)
m2 = 1.0 / (1.0 + q)

f_start = 0.018 

chi1_mag, chi1_theta, chi1_phi = 0.8, np.pi/4, 0.0

chi2_mag, chi2_theta, chi2_phi = 0.6, np.pi/2, np.pi

chi1_ini = spherical_to_cartesian(chi1_mag, chi1_theta, chi1_phi)
chi2_ini = spherical_to_cartesian(chi2_mag, chi2_theta, chi2_phi)

S_ini_tot = geometric_entropy(m1, chi1_ini) + geometric_entropy(m2, chi2_ini)

In [1]:
import sys
import numpy as np
import scipy
import gwsurrogate

print(f"Percorso Python: {sys.executable}")
print(f"Versione NumPy: {np.__version__}")
print(f"Versione SciPy: {scipy.__version__}")

# Se arrivi qui senza errori, il surrogate è pronto
sur = gwsurrogate.LoadSurrogate('NRSur7dq4')
print("--- AMBIENTE CONFIGURATO CORRETTAMENTE ---")

/home/giacomo/Thesis/Entropy-conjecture/venv310/lib/python3.10/site-packages/gwtools/const.py:52: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


lal.MSUN_SI != Msun
Percorso Python: /home/giacomo/Thesis/Entropy-conjecture/venv310/bin/python
Versione NumPy: 1.26.4
Versione SciPy: 1.11.4
Surrogate data not found for NRSur7dq4. Downloading now.
Loaded NRSur7dq4 model
--- AMBIENTE CONFIGURATO CORRETTAMENTE ---


In [2]:
import numpy as np
import gwsurrogate
import surfinBH
import matplotlib.pyplot as plt

# 1. Caricamento Modelli
sur = gwsurrogate.LoadSurrogate('NRSur7dq4')
fit = surfinBH.LoadFits('NRSur7dq4Remnant')

# 2. Parametri Fisici (i tuoi dati iniziali)
q = 2.0
chi1 = [0.2, 0.1, 0.5]
chi2 = [-0.1, 0.2, 0.3]

# 3. Estrazione Dinamica NR (Sfruttiamo il nuovo ambiente!)
# Usiamo f_low=0.01 per una spirale pulita e stabile
t, h, dyn = sur(q, chi1, chi2, f_low=0.01, dt=0.1)

chi1_t = dyn['chi1']
chi2_t = dyn['chi2']
r_t = dyn['coord_separation']

# 4. Calcolo dell'Entropia Previsionale
entropy_vals = []

for i in range(len(t)):
    # Otteniamo massa e spin del remnant previsto a quell'istante
    mf, chif, _, _, _, _ = fit.all(q, chi1_t[i], chi2_t[i])
    
    # Formula di Bekenstein-Hawking (S = Area/4)
    chif_mag = np.linalg.norm(chif)
    Sf = 2 * np.pi * (mf**2) * (1 + np.sqrt(1 - chif_mag**2))
    entropy_vals.append(Sf)

# 5. Visualizzazione Risultati
plt.figure(figsize=(12, 5))

# Plot Separazione
plt.subplot(1, 2, 1)
plt.plot(t, r_t, color='black')
plt.title("Evoluzione della Separazione (NR)")
plt.xlabel("Tempo ($M$)")
plt.ylabel("r ($M$)")

# Plot Entropia
plt.subplot(1, 2, 2)
plt.plot(t, entropy_vals, color='red', label='Entropia Finale attesa')
plt.title("Previsione Entropia del Remnant")
plt.xlabel("Tempo ($M$)")
plt.ylabel("$S_{BH}$")

plt.tight_layout()
plt.show()

print(f"Dati estratti con successo! Numero punti: {len(t)}")

Loaded NRSur7dq4 model
Loaded NRSur7dq4Remnant fit.


TypeError: 'NoneType' object is not subscriptable

In [3]:
import lalsimulation as lalsim
import scipy.integrate
import gwsurrogate

# 1. Verifica se l'integratore ODE di SciPy è davvero "vivo"
try:
    test_ode = scipy.integrate.ode(lambda t, y: y).set_integrator('vode')
    print("✅ SciPy ODE (vode) è presente e funzionante.")
except Exception as e:
    print(f"❌ Errore SciPy: {e}")

# 2. Verifica se LALSimulation è accessibile
try:
    # Una costante a caso per vedere se risponde
    print(f"✅ LALSimulation caricata. Versione: {lalsim.GetVersionString()}")
except Exception as e:
    print("❌ LALSuite non sembra installata correttamente nell'ambiente venv310.")

✅ SciPy ODE (vode) è presente e funzionante.
❌ LALSuite non sembra installata correttamente nell'ambiente venv310.


In [1]:
import lalsimulation as lalsim
import gwsurrogate
import numpy as np

# Verifica rapida
print(f"LALSimulation caricata con successo!")

# Caricamento modello
sur = gwsurrogate.LoadSurrogate('NRSur7dq4')

# I tuoi parametri
q = 2.0
chi1 = [0.2, 0.1, 0.5]
chi2 = [-0.1, 0.2, 0.3]

print("Integrazione NR in corso...")

# Chiamata al surrogate - ora dovrebbe avere i 'muscoli' per rispondere
t, h, dyn = sur(q, chi1, chi2, f_low=0.012)

if dyn is not None:
    print("--- BINGO! SURROGATE OPERATIVO ---")
    r_t = dyn['coord_separation']
    print(f"La separazione orbitale inizia a {r_t[0]:.2f} M")
    # Ora puoi finalmente calcolare l'entropia!
else:
    print("Ancora None? Controlla se 'pip list' mostra effettivamente lalsuite.")

/home/giacomo/Thesis/Entropy-conjecture/venv310/lib/python3.10/site-packages/lalsimulation/_lalsimulation_swig.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


lal.MSUN_SI != Msun
LALSimulation caricata con successo!
Loaded NRSur7dq4 model
Integrazione NR in corso...
Ancora None? Controlla se 'pip list' mostra effettivamente lalsuite.


In [2]:
try:
    import lalsimulation as lalsim
    print(f"✅ LALSimulation caricata correttamente!")
    print(f"Versione: {lalsim.GetVersionString()}")
except ImportError:
    print("❌ LALSimulation NON TROVATA. L'integratore del surrogate non può partire.")
except Exception as e:
    print(f"❌ Errore nel caricamento delle librerie C: {e}")

✅ LALSimulation caricata correttamente!
❌ Errore nel caricamento delle librerie C: module 'lalsimulation' has no attribute 'GetVersionString'


In [3]:
import lalsimulation as lalsim
import gwsurrogate
import numpy as np

# 1. Verifica REALE di LAL
print(f"✅ LALSimulation caricata. Percorso: {lalsim.__file__}")

# 2. Caricamento modello (già scaricato)
sur = gwsurrogate.LoadSurrogate('NRSur7dq4')

# 3. Parametri 'Safe' per il primo test
# Usiamo una frequenza f_low=0.015 (molto stabile nel regime NR)
q = 2.0
chi1 = [0.2, 0.1, 0.5]
chi2 = [-0.1, 0.2, 0.3]

print("Integrazione della precessione in corso...")

try:
    # Chiamata al surrogate senza dt (lascia che scelga l'integratore)
    t, h, dyn = sur(q, chi1, chi2, f_low=0.015)
    
    if dyn is not None:
        print("--- BINGO! IL SURROGATE È VIVO! ---")
        # Estraiamo i dati per la tesi
        chi1_t = dyn['chi1']
        r_t = dyn['coord_separation']
        print(f"Dinamica estratta: {len(t)} punti temporali.")
        print(f"Separazione finale raggiunta: {r_t[-1]:.2f} M")
    else:
        print("❌ L'integratore ha restituito ancora None. Tentiamo f_low ancora più alta (0.02)")
        t, h, dyn = sur(q, chi1, chi2, f_low=0.02)
        if dyn is not None:
             print("--- BINGO! (Funziona a f_low=0.02) ---")
        else:
             print("Ancora niente. Il problema è nella comunicazione tra SciPy e LAL.")
except Exception as e:
    print(f"❌ Errore critico: {e}")

✅ LALSimulation caricata. Percorso: /home/giacomo/Thesis/Entropy-conjecture/venv310/lib/python3.10/site-packages/lalsimulation/__init__.py
Loaded NRSur7dq4 model
Integrazione della precessione in corso...
❌ L'integratore ha restituito ancora None. Tentiamo f_low ancora più alta (0.02)
Ancora niente. Il problema è nella comunicazione tra SciPy e LAL.


In [4]:
import sys
# Forza il caricamento di imp se non presente
try:
    import imp
except ImportError:
    import zombie_imp as imp

import numpy as np
import gwsurrogate
import lalsimulation as lalsim

# Caricamento Modello
sur = gwsurrogate.LoadSurrogate('NRSur7dq4')

# Parametri 'Ultra-Stabili'
q = 1.5              # Rapporto di massa semplice
chi1 = [0, 0, 0.2]   # Spin solo su Z (no precessione complessa per il test)
chi2 = [0, 0, -0.2]

print("--- TENTATIVO DI EMERGENZA ---")

# Proviamo a integrare solo gli ultimi istanti (f_low molto alta = 0.03)
# Questo riduce il lavoro dell'integratore del 90%
try:
    t, h, dyn = sur(q, chi1, chi2, f_low=0.03)
    
    if dyn is not None:
        print("✅ BINGO! Il surrogate ha risposto.")
        print(f"Punti estratti: {len(t)}")
    else:
        print("❌ Ancora None. L'integratore VODE di SciPy sta fallendo silenziosamente.")
        print("\nDIAGNOSI FINALE PER LA TESI:")
        print("Il problema è nell'interfaccia Fortran/C del tuo sistema WSL.")
except Exception as e:
    print(f"❌ Errore durante l'integrazione: {e}")

/tmp/ipykernel_29451/4188194274.py:4: DeprecationWarning: the imp module is deprecated in favour of importlib and slated for removal in Python 3.12; see the module's documentation for alternative uses
  import imp


Loaded NRSur7dq4 model
--- TENTATIVO DI EMERGENZA ---
❌ Ancora None. L'integratore VODE di SciPy sta fallendo silenziosamente.

DIAGNOSI FINALE PER LA TESI:
Il problema è nell'interfaccia Fortran/C del tuo sistema WSL.
